# Initialization

In [ ]:
from __future__ import annotations

import sys
import ast
import math
import os
import re
import tempfile
import torch
import pytesseract
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps
from tqdm.auto import tqdm



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    s

In [22]:
# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------

INPUT_CSV = "/home/yishin/keith/patent_research/model_io/Impact_Sub_KW.csv"
OUTPUT_CSV = "/home/yishin/keith/patent_research/model_io/Impact_Sub_KW_IMG.csv"
OUTPUT_ROOT = "image_crops"

TITLE_COL = "title"
IMAGE_PATHS_COL = "image_paths"

USE_GPU = True
ROTATION_ANGLES = (0, 90, 180, 270)
MIN_HEIGHT = 3000
JPEG_QUALITY = 95

MIN_AREA = 500
PADDING = 25
MERGE_GAP = 35
INVERT = True
MERGE_TO_FIG_COUNT = True
DETECT_LABELS = True


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_COLORS: List[Tuple[int, int, int]] = [
    (230,  90,  50),
    ( 50, 170,  90),
    ( 80, 120, 220),
    (190,  60, 160),
    (200, 160,  30),
    ( 40, 180, 180),
    (220,  60, 100),
    (100,  80, 220),
]

_FIG_PATTERN = re.compile(r"\bF[I1L]G(?:[\.\s]*\d+)?\b", re.IGNORECASE)
_FIG_NUM_PATTERN = re.compile(r"\bF[I1L]G[\.\s]*(\d+)\b", re.IGNORECASE)


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def try_literal_eval(value):
    """
    Safely parse a stringified Python literal such as a list of image paths.
    Returns the original value unchanged if parsing fails.
    """
    if isinstance(value, str):
        stripped = value.strip()
        if stripped and stripped[0] in ("[", "{", "(") and stripped[-1] in ("]", "}", ")"):
            try:
                return ast.literal_eval(stripped)
            except (ValueError, SyntaxError):
                pass
    return value


def _sanitize_folder_name(name: str) -> str:
    """Replace filesystem-unsafe characters with underscores."""
    return re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name).strip()


def load_dataframe(csv_path: str) -> pd.DataFrame:
    """Load the CSV and restore stringified lists/dicts back to Python objects."""
    return pd.read_csv(csv_path).map(try_literal_eval)


def get_path_list(value) -> List[str]:
    """Normalize a dataframe cell into a list of image paths."""
    value = value if isinstance(value, list) else try_literal_eval(value)
    if not value:
        return []
    return [str(p) for p in value]


df = load_dataframe(INPUT_CSV)
df.head()


,title,caption,image_paths,Loc_class,main_class,sub_class,keywords
0,Animal kibble,The image is a white square with a diamond sha...,[impact_dataset/2022/USD0942111-20220201/USD09...,"{11-03, 01-06}",1,6,"[food dish, bowl, container, lid, label]"
1,Animal kibble,"The image is a white, three-dimensional, geome...",[impact_dataset/2022/USD0948836-20220419/USD09...,"{11-03, 01-06}",1,6,"[body, ears, eyes, nose, mouth, paws, tail]"
2,Packaged foodstuff,The image is a square-shaped drawing of packag...,[impact_dataset/2022/USD0959087-20220802/USD09...,{01-01},1,1,"[packaging, container, label, lid, cap, tube, ..."
3,Freeze dried citrus fruit product,The image is a white drawing of a freeze dried...,[impact_dataset/2022/USD0940991-20220118/USD09...,{01-01},1,1,"[fruit, peel, pulp, membrane, water]"
4,Rolled pet treat,"The image is a rolled pet treat, which is a ty...",[impact_dataset/2022/USD0973298-20221227/USD09...,{01-01},1,1,"[plastic container, treat, kibble, handle, bag]"


# Fig label detection

In [23]:
def _preprocess_for_ocr(
    image: Image.Image,
    min_height: int,
    threshold: int = 180,
    dilate: bool = True,
) -> Image.Image:
    """
    Strong OCR preprocessing for thin patent drawings and vertical FIG labels.
    """
    image = image.convert("L")
    image = ImageOps.autocontrast(image)

    w, h = image.size
    scale = max(1.0, min_height / h)

    if scale > 1.0:
        image = image.resize(
            (int(w * scale), int(h * scale)),
            Image.Resampling.LANCZOS,
        )

    arr = np.array(image)
    arr = np.where(arr < threshold, 0, 255).astype(np.uint8)

    if dilate:
        kernel = np.ones((2, 2), np.uint8)
        arr = cv2.dilate(arr, kernel, iterations=1)

    return Image.fromarray(arr)


def count_fig_labels_tesseract(
    image_path: str,
    min_height: int,
    psm_modes: Tuple[int, ...] = (6, 11, 3, 12, 5),
    rotation_angles: Tuple[int, ...] = (0, 90, 180, 270),
) -> Tuple[int, int]:
    """
    Tesseract OCR version.

    Returns:
        (best_fig_count, best_rotation_angle)

    It checks:
    1. full image
    2. left strip
    3. bottom strip

    This helps detect vertical FIG labels near margins.
    """
    try:
        with Image.open(image_path) as img:
            source = img.convert("RGB").copy()
    except Exception as err:
        tqdm.write(f"[OCR Error] {image_path}: {err}")
        return 0, 0

    best_count = 0
    best_rotation = 0

    tess_config_base = (
        "-c tessedit_char_whitelist=FIGfig.0123456789 "
        "-c preserve_interword_spaces=1"
    )

    for angle in rotation_angles:
        rotated = source.rotate(angle, expand=True) if angle != 0 else source

        w, h = rotated.size

        candidates = [
            rotated,
            rotated.crop((0, 0, int(w * 0.25), h)),
            rotated.crop((0, int(h * 0.75), w, h)),
        ]

        for candidate in candidates:
            processed = _preprocess_for_ocr(
                candidate,
                min_height=min_height,
                threshold=190,
                dilate=True,
            )

            for psm in psm_modes:
                try:
                    text = pytesseract.image_to_string(
                        processed,
                        config=f"--psm {psm} {tess_config_base}",
                    )
                except Exception as err:
                    tqdm.write(
                        f"[Tesseract Error] {image_path} angle={angle} psm={psm}: {err}"
                    )
                    continue

                normalized_text = " ".join(text.split())
                count = len(_FIG_PATTERN.findall(normalized_text))

                if count > best_count:
                    best_count = count
                    best_rotation = angle

    return best_count, best_rotation


def detect_fig_label_tesseract(
    crop_rgb: np.ndarray,
    min_height: int = 1000,
    psm_modes: Tuple[int, ...] = (6, 11, 3, 12, 5),
) -> Optional[str]:
    """
    Detect FIG number from a crop using Tesseract.

    Returns:
        Fig_1, Fig_2, etc.
    """
    pil_img = Image.fromarray(crop_rgb)

    tess_config_base = (
        "-c tessedit_char_whitelist=FIGfig.0123456789 "
        "-c preserve_interword_spaces=1"
    )

    for angle in (0, 90, 180, 270):
        rotated = pil_img.rotate(angle, expand=True) if angle != 0 else pil_img

        processed = _preprocess_for_ocr(
            rotated,
            min_height=min_height,
            threshold=190,
            dilate=True,
        )

        for psm in psm_modes:
            try:
                text = pytesseract.image_to_string(
                    processed,
                    config=f"--psm {psm} {tess_config_base}",
                )
            except Exception:
                continue

            normalized = " ".join(text.split())
            match = _FIG_NUM_PATTERN.search(normalized)

            if match:
                return f"Fig_{match.group(1)}"

    return None


def _unique_output_path(output_dir: str, stem: str) -> str:
    """Return a non-colliding .jpg path inside output_dir."""
    candidate = os.path.join(output_dir, f"{stem}.jpg")
    if not os.path.exists(candidate):
        return candidate

    counter = 1
    while True:
        candidate = os.path.join(output_dir, f"{stem}_{counter}.jpg")
        if not os.path.exists(candidate):
            return candidate
        counter += 1


def _filter_images_with_fig(
    image_paths: List[str],
    output_dir: str,
    min_height: int,
    psm_modes: Tuple[int, ...] = (6, 11, 3, 12, 5),
    rotation_angles: Tuple[int, ...] = (0, 90, 180, 270),
    jpeg_quality: int = 95,
    show_progress: bool = True,
) -> Tuple[List[str], Dict[str, int], Dict[str, int], Dict[str, str]]:
    """
    Keep only images containing at least one FIG-like label.

    The best detected rotation is applied to the temporary JPEG, so all later
    bbox detection and final crops use the correct orientation.
    """
    os.makedirs(output_dir, exist_ok=True)

    retained: List[str] = []
    fig_counts: Dict[str, int] = {}
    rotations: Dict[str, int] = {}
    original_by_filtered: Dict[str, str] = {}

    iterator = tqdm(image_paths, desc="  Fig label detection", disable=not show_progress)

    for image_path in iterator:
        image_path = str(image_path)

        if not os.path.isfile(image_path):
            tqdm.write(f"  [Warning] File not found: {image_path}")
            fig_counts[image_path] = 0
            rotations[image_path] = 0
            continue

        count, rotation = count_fig_labels_tesseract(
            image_path=image_path,
            min_height=min_height,
            psm_modes=psm_modes,
            rotation_angles=rotation_angles,
        )

        fig_counts[image_path] = count
        rotations[image_path] = rotation

        if count == 0:
            continue

        stem = os.path.splitext(os.path.basename(image_path))[0]
        out_path = _unique_output_path(output_dir, stem)

        try:
            with Image.open(image_path) as img:
                image = img.convert("RGB")
                if rotation != 0:
                    image = image.rotate(rotation, expand=True)
                image.save(out_path, format="JPEG", quality=jpeg_quality)

            retained.append(out_path)
            original_by_filtered[out_path] = image_path

        except Exception as err:
            tqdm.write(f"  [Save Error] {image_path}: {err}")

    return retained, fig_counts, rotations, original_by_filtered


In [ ]:
filtered_paths, fig_counts, rotations, original_by_filtered = _filter_images_with_fig(
    image_paths=df.iloc[[10]].iloc[0]["image_paths"],
    output_dir="tmp_fig_filtered",
    min_height=3000,
    psm_modes=(6, 11, 3, 12, 5),
    rotation_angles=(0, 90, 270),
    jpeg_quality=95,
    show_progress=True,
)

  Fig label detection: 100%|██████████| 6/6 [05:30<00:00, 55.14s/it]


In [25]:
filtered_paths, fig_counts, rotations, original_by_filtered

(['tmp_fig_filtered/USD0949543-20220426-D00000.jpg',
  'tmp_fig_filtered/USD0949543-20220426-D00001.jpg',
  'tmp_fig_filtered/USD0949543-20220426-D00002.jpg',
  'tmp_fig_filtered/USD0949543-20220426-D00003.jpg',
  'tmp_fig_filtered/USD0949543-20220426-D00004.jpg',
  'tmp_fig_filtered/USD0949543-20220426-D00005.jpg'],
 {'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00000.TIF': 1,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00001.TIF': 1,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00002.TIF': 1,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00003.TIF': 2,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00004.TIF': 1,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00005.TIF': 1},
 {'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00000.TIF': 90,
  'impact_dataset/2022/USD0949543-20220426/USD0949543-20220426-D00001.TIF': 180,
  'impact_dataset/2022/USD0949543-20220426/US

# Whitespace division

In [ ]:
def _crop_by_whitespace(
    image_paths: List[str],
    min_area: int,
    padding: int,
    merge_gap: int,
    invert: bool,
    show_progress: bool,
    rotations: Optional[Dict[str, int]] = None,
    original_by_filtered: Optional[Dict[str, str]] = None,
) -> Dict[str, Dict]:
    """
    Detect and crop figures from already-oriented patent images.
    """
    results: Dict[str, Dict] = {}

    iterator = tqdm(image_paths, desc="  Whitespace division", disable=not show_progress)

    for image_path in iterator:
        img = cv2.imread(image_path)

        if img is None:
            results[image_path] = {
                "boxes": [],
                "crops": [],
                "rotation": 0,
                "error": "Could not read image",
            }
            continue

        original_path = (
            original_by_filtered.get(image_path, image_path)
            if original_by_filtered
            else image_path
        )

        rotation = rotations.get(original_path, 0) if rotations else 0

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)

        thresh_type = cv2.THRESH_BINARY_INV if invert else cv2.THRESH_BINARY
        binary = cv2.threshold(
            blurred,
            0,
            255,
            thresh_type + cv2.THRESH_OTSU,
        )[1]

        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (merge_gap, merge_gap))
        dilated = cv2.dilate(binary, kernel, iterations=1)

        contours, _ = cv2.findContours(
            dilated,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        h_img, w_img = img.shape[:2]
        boxes: List[Tuple[int, int, int, int]] = []

        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            if w * h < min_area:
                continue

            x1 = max(x - padding, 0)
            y1 = max(y - padding, 0)
            x2 = min(x + w + padding, w_img)
            y2 = min(y + h + padding, h_img)
            boxes.append((x1, y1, x2 - x1, y2 - y1))

        boxes.sort(key=lambda b: (b[1], b[0]))

        crops = [
            cv2.cvtColor(img[y:y + h, x:x + w], cv2.COLOR_BGR2RGB)
            for x, y, w, h in boxes
        ]

        results[image_path] = {
            "boxes": boxes,
            "crops": crops,
            "rotation": rotation,
            "original_path": original_path,
        }

    return results


def _save_crops(
    crop_arrays: List[np.ndarray],
    source_stem: str,
    output_dir: str,
    jpeg_quality: int,
    detect_labels: bool = True,
) -> List[str]:
    """
    Save RGB crop arrays to output_dir.

    If detect_labels=True, saved file names use detected labels such as Fig_1.jpg
    when possible. Otherwise, the fallback name is {source_stem}_{idx:02d}.jpg.
    """
    saved: List[str] = []
    used_names: set = set()
    os.makedirs(output_dir, exist_ok=True)

    for idx, crop in enumerate(crop_arrays, start=1):
        label = detect_fig_label_tesseract(crop) if detect_labels else None
        filename = (
            f"{label}.jpg"
            if (label and f"{label}.jpg" not in used_names)
            else f"{source_stem}_{idx:02d}.jpg"
        )
        used_names.add(filename)

        out_path = os.path.join(output_dir, filename)
        Image.fromarray(crop).save(out_path, format="JPEG", quality=jpeg_quality)
        saved.append(out_path)

    return saved


# Merge bounding boxes

In [ ]:
def merge_figure_boxes(
    boxes: List[Tuple[int, int, int, int]],
    box_count: int,
    mode: str = "edge",
) -> List[Tuple[int, int, int, int]]:
    """
    Greedily merge bounding boxes until exactly box_count remain.
    """
    if box_count <= 0:
        raise ValueError("box_count must be a positive integer.")
    if len(boxes) <= box_count:
        return list(boxes)

    def _edge_dist(a: tuple, b: tuple) -> float:
        dx = max(0, max(a[0], b[0]) - min(a[0] + a[2], b[0] + b[2]))
        dy = max(0, max(a[1], b[1]) - min(a[1] + a[3], b[1] + b[3]))
        return math.hypot(dx, dy)

    def _center_dist(a: tuple, b: tuple) -> float:
        return math.hypot(
            (a[0] + a[2] / 2) - (b[0] + b[2] / 2),
            (a[1] + a[3] / 2) - (b[1] + b[3] / 2),
        )

    def _union(a: tuple, b: tuple) -> Tuple[int, int, int, int]:
        x = min(a[0], b[0])
        y = min(a[1], b[1])
        x2 = max(a[0] + a[2], b[0] + b[2])
        y2 = max(a[1] + a[3], b[1] + b[3])
        return (x, y, x2 - x, y2 - y)

    dist_fn = _edge_dist if mode == "edge" else _center_dist
    pool = list(boxes)

    while len(pool) > box_count:
        best_i, best_j, best_d = 0, 1, float("inf")

        for i in range(len(pool)):
            for j in range(i + 1, len(pool)):
                d = dist_fn(pool[i], pool[j])
                if d < best_d:
                    best_d, best_i, best_j = d, i, j

        merged = _union(pool[best_i], pool[best_j])
        pool = [b for k, b in enumerate(pool) if k not in (best_i, best_j)]
        pool.append(merged)

    pool.sort(key=lambda b: (b[1], b[0]))
    return pool


def _apply_box_merge(
    image_path: str,
    boxes: List[Tuple[int, int, int, int]],
    fig_count: int,
) -> Tuple[List[Tuple[int, int, int, int]], List[np.ndarray]]:
    """
    Merge boxes down to fig_count and re-extract RGB crops from image_path.
    image_path should already point to the correctly oriented temporary image.
    """
    if fig_count <= 0 or len(boxes) <= fig_count:
        return boxes, []

    merged_boxes = merge_figure_boxes(boxes, box_count=fig_count)

    img = cv2.imread(image_path)
    if img is None:
        return boxes, []

    crops = [
        cv2.cvtColor(img[y:y + h, x:x + w], cv2.COLOR_BGR2RGB)
        for x, y, w, h in merged_boxes
    ]

    return merged_boxes, crops


def process_dataframe(
    df: pd.DataFrame,
    output_root: str = "image_crops",
    title_col: str = "title",
    image_paths_col: str = "image_paths",
    min_height: int = 3000,
    rotation_angles: Tuple[int, ...] = (0, 90, 180, 270),
    jpeg_quality: int = 95,
    min_area: int = 500,
    padding: int = 25,
    merge_gap: int = 35,
    invert: bool = True,
    merge_to_fig_count: bool = True,
    detect_labels: bool = True,
    show_progress: bool = True,
) -> Dict[str, Dict]:
    """
    Run the full DataFrame-driven figure-cropping pipeline.

    This function preserves DataFrame usage: it reads image paths from each row,
    saves cropped images, then writes the saved paths into df["cropped_paths"].
    """
    results_by_title: Dict[str, Dict] = {}
    paths_by_index: Dict[int, List[str]] = {}

    rows = list(df.iterrows())
    row_bar = tqdm(
        total=len(rows),
        desc="Patents",
        unit="patent",
        position=0,
        leave=True,
        dynamic_ncols=True,
        disable=not show_progress,
    )

    for idx, row in rows:
        title = str(row[title_col])
        image_paths = get_path_list(row[image_paths_col])
        paths_by_index[idx] = []

        if show_progress:
            row_bar.set_postfix(patent=title[:45], refresh=False)

        if not image_paths:
            tqdm.write(f"[Skip] No image paths for: {title!r}")
            results_by_title[title] = {}
            row_bar.update(1)
            continue

        safe_title = _sanitize_folder_name(title)
        title_dir = os.path.join(output_root, f"{idx}_{safe_title}")
        os.makedirs(title_dir, exist_ok=True)

        with tempfile.TemporaryDirectory() as tmp_filtered:
            filtered_paths, fig_counts, rotations, original_by_filtered = _filter_images_with_fig(
                image_paths=image_paths,
                output_dir=tmp_filtered,
                min_height=min_height,
                rotation_angles=rotation_angles,
                jpeg_quality=jpeg_quality,
                show_progress=False,
            )

            if not filtered_paths:
                tqdm.write(f"  [Skip] No FIG images found for: {title!r}")
                results_by_title[title] = {}
                row_bar.update(1)
                continue

            crop_results = _crop_by_whitespace(
                image_paths=filtered_paths,
                min_area=min_area,
                padding=padding,
                merge_gap=merge_gap,
                invert=invert,
                show_progress=False,
                rotations=rotations,
                original_by_filtered=original_by_filtered,
            )

            count_by_stem: Dict[str, int] = {
                os.path.splitext(os.path.basename(p))[0]: cnt
                for p, cnt in fig_counts.items()
            }

            if merge_to_fig_count:
                for filtered_path, data in crop_results.items():
                    stem = os.path.splitext(os.path.basename(filtered_path))[0]
                    base_stem = re.sub(r"_\d+$", "", stem)
                    fig_count = count_by_stem.get(stem, count_by_stem.get(base_stem, 0))

                    if fig_count > 0 and len(data["boxes"]) > fig_count:
                        merged_boxes, merged_crops = _apply_box_merge(
                            image_path=filtered_path,
                            boxes=data["boxes"],
                            fig_count=fig_count,
                        )
                        if merged_crops:
                            data["boxes"] = merged_boxes
                            data["crops"] = merged_crops

            row_summary: Dict[str, Dict] = {}

            for filtered_path, data in crop_results.items():
                stem = os.path.splitext(os.path.basename(filtered_path))[0]
                base_stem = re.sub(r"_\d+$", "", stem)
                fig_count = count_by_stem.get(stem, count_by_stem.get(base_stem, 0))

                saved_paths = _save_crops(
                    crop_arrays=data["crops"],
                    source_stem=base_stem,
                    output_dir=title_dir,
                    jpeg_quality=jpeg_quality,
                    detect_labels=detect_labels,
                )

                row_summary[base_stem] = {
                    "fig_count": fig_count,
                    "rotation": data.get("rotation", 0),
                    "original_path": data.get("original_path"),
                    "boxes": data["boxes"],
                    "saved_crops": saved_paths,
                }

                tqdm.write(f"  ✓ {base_stem}: {len(saved_paths)} crop(s)")

        results_by_title[title] = row_summary
        paths_by_index[idx] = [p for v in row_summary.values() for p in v["saved_crops"]]
        row_bar.update(1)

    row_bar.close()

    df["cropped_paths"] = pd.Series(paths_by_index)
    return results_by_title


In [ ]:
# Run the pipeline from START_ROW onward while preserving the original df.

df_part = df.iloc[START_ROW:].copy()

results = process_dataframe(
    df_part,
    output_root=OUTPUT_ROOT,
    title_col=TITLE_COL,
    image_paths_col=IMAGE_PATHS_COL,
    min_height=MIN_HEIGHT,
    rotation_angles=ROTATION_ANGLES,
    jpeg_quality=JPEG_QUALITY,
    min_area=MIN_AREA,
    padding=PADDING,
    merge_gap=MERGE_GAP,
    invert=INVERT,
    merge_to_fig_count=MERGE_TO_FIG_COUNT,
    detect_labels=DETECT_LABELS,
    show_progress=True,
)

# Carry cropped paths back to the original dataframe.
df.loc[df_part.index, "cropped_paths"] = df_part["cropped_paths"]

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved updated dataframe to: {OUTPUT_CSV}")
